# Notebook 04 — Modelo de Predição de Voto

**Sprint 3 — Lei e Política**

Modelo supervisionado para testar a hipótese central do projeto: **prever o voto
individual de um deputado (favorável/contrário) com acurácia ≥ 70% em split temporal**.

Escopo: **Câmara** (só há votos nominais da Câmara; o Senado coletou apenas matérias).

## Pipeline

1. Montagem do dataset voto-a-voto (junção `votos`×`votacoes`×`proposicoes`×`parlamentares`)
2. Split temporal 80/20 por data da votação (sem vazamento)
3. Features: `partido`, `uf`, `tema_cluster`, `mes` + tendência partido×tema (derivada só do treino)
4. RandomForestClassifier — treino e avaliação (acurácia, F1 macro, matriz de confusão, importâncias)
5. Gravação dos resultados em `metricas_modelo`

In [1]:
import sys
sys.path.insert(0, '..')

import logging

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

from src.db import buscar_todos, inserir_metricas

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
log = logging.getLogger('04_modelo')
print('Módulos carregados.')

Módulos carregados.


## 1. Montagem do dataset voto-a-voto

Cada linha é um voto nominal de um deputado em uma votação. Juntamos `votos` →
`votacoes` (data, proposição) → `proposicoes` (tema_cluster) → `parlamentares`
(partido, uf). Mantemos apenas votos decisivos (`favoravel`/`contrario`).

In [2]:
# Carrega as 4 tabelas (escopo: Câmara — só há votos nominais da Câmara)
votos = pd.DataFrame(buscar_todos('votos', 'votacao_id,parlamentar_id,voto'))
votacoes = pd.DataFrame(buscar_todos('votacoes', 'id,proposicao_id,data'))
proposicoes = pd.DataFrame(buscar_todos('proposicoes', 'id,tema_cluster'))
parlamentares = pd.DataFrame(buscar_todos('parlamentares', 'id,partido,uf,casa'))

print(f'votos={len(votos)}  votacoes={len(votacoes)}  '
      f'proposicoes={len(proposicoes)}  parlamentares={len(parlamentares)}')

# Chaves de junção como inteiro anulável (proposicao_id vem como object por conter nulos)
for d, col in [(votos, 'votacao_id'), (votos, 'parlamentar_id'),
               (votacoes, 'id'), (votacoes, 'proposicao_id'),
               (proposicoes, 'id'), (parlamentares, 'id')]:
    d[col] = pd.to_numeric(d[col], errors='coerce').astype('Int64')

# Junções voto-a-voto
df = votos.merge(votacoes, left_on='votacao_id', right_on='id', suffixes=('', '_vt'))
df = df.merge(proposicoes, left_on='proposicao_id', right_on='id',
              how='left', suffixes=('', '_pr'))
df = df.merge(parlamentares, left_on='parlamentar_id', right_on='id',
              how='left', suffixes=('', '_pl'))

# Apenas Câmara e voto binário decisivo (abstenções fora — hipótese é binária)
df = df[df['casa'] == 'camara']
df = df[df['voto'].isin(['favoravel', 'contrario'])].copy()

# tema_cluster ausente (votação sem proposição linkada) -> categoria 'sem_tema' (-1)
df['tema_cluster'] = df['tema_cluster'].fillna(-1).astype(int)

# data -> datetime; descarta linhas sem data (necessárias para o split temporal)
df['data'] = pd.to_datetime(df['data'], errors='coerce')
df = df[df['data'].notna()].copy()

# preenche partido/uf ausentes para não perder linhas no one-hot
df['partido'] = df['partido'].fillna('DESCONHECIDO')
df['uf'] = df['uf'].fillna('XX')

# alvo binário: 1 = favoravel, 0 = contrario
df['alvo'] = (df['voto'] == 'favoravel').astype(int)

print(f'\nDataset final: {len(df)} votos')
print('Distribuição de classes:')
print(df['voto'].value_counts())

2026-06-23 23:06:41,520 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/votos?select=votacao_id%2Cparlamentar_id%2Cvoto&offset=0&limit=1000 "HTTP/2 200 OK"
2026-06-23 23:06:41,802 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/votos?select=votacao_id%2Cparlamentar_id%2Cvoto&offset=1000&limit=1000 "HTTP/2 200 OK"
2026-06-23 23:06:42,062 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/votos?select=votacao_id%2Cparlamentar_id%2Cvoto&offset=2000&limit=1000 "HTTP/2 200 OK"
2026-06-23 23:06:42,253 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/votos?select=votacao_id%2Cparlamentar_id%2Cvoto&offset=3000&limit=1000 "HTTP/2 200 OK"
2026-06-23 23:06:42,465 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/votos?select=votacao_id%2Cparlamentar_id%2Cvoto&offset=4000&limit=1000 "HTTP/2 200 OK"
2026-06-23 23:06:42,644 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.

votos=55570  votacoes=137  proposicoes=22041  parlamentares=726

Dataset final: 55233 votos
Distribuição de classes:
voto
favoravel    35206
contrario    20027
Name: count, dtype: int64


## 2. Split temporal

Ordenamos os votos pela data da votação e cortamos no **percentil 80** das datas:
treino = votações mais antigas, teste = mais recentes. Isso simula prever votos futuros a
partir do histórico — o split aleatório seria irrealista e otimista.

In [3]:
# Split temporal 80/20 por data da votação (sem vazamento)
df = df.sort_values('data').reset_index(drop=True)
data_corte = df['data'].quantile(0.8)

treino = df[df['data'] < data_corte].copy()
teste = df[df['data'] >= data_corte].copy()

print(f'data_corte = {data_corte.date()}')
print(f'n_treino = {len(treino):>6}  ({treino["data"].min().date()} → {treino["data"].max().date()})')
print(f'n_teste  = {len(teste):>6}  ({teste["data"].min().date()} → {teste["data"].max().date()})')
print('\nClasses no treino:', treino['voto'].value_counts().to_dict())
print('Classes no teste :', teste['voto'].value_counts().to_dict())

assert treino['alvo'].nunique() == 2 and teste['alvo'].nunique() == 2, \
    'Treino ou teste sem as duas classes — ajustar o corte temporal.'

data_corte = 2026-04-29
n_treino =  44043  (2025-02-11 → 2026-04-28)
n_teste  =  11190  (2026-04-29 → 2026-06-03)

Classes no treino: {'favoravel': 27415, 'contrario': 16628}
Classes no teste : {'favoravel': 7791, 'contrario': 3399}


## 3. Feature engineering

Features categóricas (`partido`, `uf`, `tema_cluster`, `mes`) via one-hot + uma feature
numérica: `tend_partido_tema` (proporção histórica de votos favoráveis daquele partido
naquele tema). **Crítico**: essa tendência é calculada **somente no treino** — usá-la sobre
todo o dataset seria vazamento temporal.

In [4]:
# Feature derivada SÓ do treino: tendência do partido naquele tema (evita vazamento).
media_global = treino['alvo'].mean()
tend = (
    treino.groupby(['partido', 'tema_cluster'])['alvo']
    .mean()
    .rename('tend_partido_tema')
    .reset_index()
)

def add_features(d):
    d = d.merge(tend, on=['partido', 'tema_cluster'], how='left')
    # combinações partido×tema inéditas no teste recebem a média global do treino
    d['tend_partido_tema'] = d['tend_partido_tema'].fillna(media_global)
    d['mes'] = d['data'].dt.month
    return d

treino_f = add_features(treino)
teste_f = add_features(teste)

COLS_CAT = ['partido', 'uf', 'tema_cluster', 'mes']
COLS_NUM = ['tend_partido_tema']

def montar_X(d):
    return pd.get_dummies(d[COLS_CAT + COLS_NUM], columns=COLS_CAT)

X_treino = montar_X(treino_f)
# Realinha o teste às colunas do treino (categorias ausentes viram 0; evita vazamento de schema)
X_teste = montar_X(teste_f).reindex(columns=X_treino.columns, fill_value=0)
y_treino = treino_f['alvo'].values
y_teste = teste_f['alvo'].values

print(f'X_treino: {X_treino.shape}   X_teste: {X_teste.shape}')

X_treino: (44043, 55)   X_teste: (11190, 55)


## 4. Treino e avaliação

`RandomForestClassifier` com `class_weight='balanced'` (a classe `favoravel` domina por
causa da disciplina partidária). Reportamos **acurácia** e **F1 macro** — o F1 macro é o
indicador honesto quando há desbalanceamento.

In [5]:
modelo = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1,
)
modelo.fit(X_treino, y_treino)
pred = modelo.predict(X_teste)

acuracia = accuracy_score(y_teste, pred)
f1m = f1_score(y_teste, pred, average='macro')

print(f'Acurácia : {acuracia:.4f}')
print(f'F1 macro : {f1m:.4f}')
meta = 'ATINGIDA ✅' if acuracia >= 0.70 else 'NÃO atingida ❌'
print(f'Meta da hipótese (acurácia >= 0.70): {meta}')

print('\nMatriz de confusão (linhas=real, colunas=previsto) [0=contrario, 1=favoravel]:')
print(confusion_matrix(y_teste, pred))

print('\n' + classification_report(y_teste, pred, target_names=['contrario', 'favoravel']))

imp = pd.Series(modelo.feature_importances_, index=X_treino.columns).sort_values(ascending=False)
print('Top-15 features mais importantes:')
print(imp.head(15).to_string())

Acurácia : 0.6294
F1 macro : 0.4991
Meta da hipótese (acurácia >= 0.70): NÃO atingida ❌

Matriz de confusão (linhas=real, colunas=previsto) [0=contrario, 1=favoravel]:
[[ 668 2731]
 [1416 6375]]

              precision    recall  f1-score   support

   contrario       0.32      0.20      0.24      3399
   favoravel       0.70      0.82      0.75      7791

    accuracy                           0.63     11190
   macro avg       0.51      0.51      0.50     11190
weighted avg       0.58      0.63      0.60     11190

Top-15 features mais importantes:
tend_partido_tema    0.115894
mes_10               0.102081
mes_2                0.071470
mes_3                0.059257
mes_4                0.053462
partido_PL           0.037475
uf_RJ                0.021894
uf_SP                0.021381
mes_7                0.020974
uf_PE                0.019260
uf_SC                0.018993
uf_BA                0.018870
uf_MG                0.018556
uf_MA                0.017995
uf_PR                0.

## 5. Gravar métricas no Supabase

Registra o resultado do treino em `metricas_modelo` (append-only — mantém o histórico de runs).

In [6]:
registro = {
    'modelo': 'random_forest_v1',
    'acuracia': round(float(acuracia), 4),
    'f1_macro': round(float(f1m), 4),
    'data_corte': str(data_corte.date()),
    'n_treino': int(len(treino)),
    'n_teste': int(len(teste)),
}
inserir_metricas(registro)
print('Métricas gravadas em metricas_modelo:')
for k, v in registro.items():
    print(f'  {k}: {v}')

2026-06-23 23:07:31,799 [INFO] HTTP Request: POST https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/metricas_modelo "HTTP/2 201 Created"
2026-06-23 23:07:31,802 [INFO] inserir metricas_modelo: random_forest_v1


Métricas gravadas em metricas_modelo:
  modelo: random_forest_v1
  acuracia: 0.6294
  f1_macro: 0.4991
  data_corte: 2026-04-29
  n_treino: 44043
  n_teste: 11190
